In [ ]:
#####       合并数据集‘cleaned_sleep_data_fixed_time.csv‘
import pandas as pd
import os
import re


def extract_file_paths(file_path, folder):
    """从SHA256SUMS.txt中提取指定文件夹的文件路径"""
    paths = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                match = re.search(f'{folder}/[^ ]+', line)
                if match:
                    paths.append(match.group())
        print(f"在 {folder} 文件夹中找到 {len(paths)} 个文件")
        return paths
    except Exception as e:
        print(f"读取 {file_path} 时出错: {str(e)}")
        return []


def read_and_process_file(file_path):
    """读取并处理单个文件"""
    normalized_path = os.path.abspath(file_path)
    # print(f"正在检查路径: {normalized_path}")  # 调试路径
    try:
        if not os.path.exists(normalized_path):
            print(f"文件不存在: {normalized_path}")
            return None
        # 根据文件类型选择不同的读取方式，添加quoting参数来正确处理引号
        if 'heartrate' in normalized_path:
            df = pd.read_csv(normalized_path, sep='\t', header=None,
                             names=['time', 'heart_rate'],
                             quoting=3)  # 读取心率数据
        elif 'labeled_sleep' in normalized_path:
            df = pd.read_csv(normalized_path, sep='\t', header=None,
                             names=['time', 'sleep_state'],
                             quoting=3)  # 读取睡眠状态数据
        elif 'acceleration' in normalized_path:
            df = pd.read_csv(normalized_path, sep='\t', header=None,
                             names=['time', 'x', 'y', 'z'],
                             quoting=3)  # 读取加速度数据
        elif 'steps' in normalized_path:
            df = pd.read_csv(normalized_path, sep='\t', header=None,
                             names=['time', 'steps'],
                             quoting=3)  # 读取步数数据
        else:
            print(f"未知的文件类型: {normalized_path}")
            return None

        # 从文件名中提取用户ID
        filename = os.path.basename(normalized_path)  # 获取文件名
        user_id = re.search(r'^(\d+)_', filename).group(1)  # 提取ID部分
        df['user_id'] = user_id

        print(f"成功读取文件 {normalized_path}, 数据形状: {df.shape}, 用户ID: {user_id}")
        return df
    except Exception as e:
        print(f"处理文件 {normalized_path} 时出错: {str(e)}")
        return None


def main():
    # 定义要处理的文件夹（根据实际存在的路径调整）
    folders = ['heart_rate', 'labels', 'motion', 'steps']

    # 添加基础路径（根据实际文件存储位置调整）
    BASE_DIR = os.getcwd()

    # 为每个文件夹创建对应的DataFrame
    for folder in folders:
        print(f"\n{'='*50}")
        print(f"处理 {folder} 文件夹...")

        # 获取该文件夹下的所有文件路径
        file_paths = extract_file_paths('SHA256SUMS.txt', folder)

        if not file_paths:
            print(f"没有找到 {folder} 文件夹中的文件")
            continue

        # 读取并合并所有文件
        dfs = []
        for rel_path in file_paths:
            # 拼接绝对路径
            abs_path = os.path.join(
                BASE_DIR, rel_path.replace('/', '\\').strip('\n'))
            print(f"\n正在处理: {abs_path}")
            df = read_and_process_file(abs_path)
            if df is not None:
                dfs.append(df)

        # 合并该文件夹的所有数据
        if dfs:
            try:
                combined_df = pd.concat(dfs, ignore_index=True)
                # 保存到CSV文件
                output_file = f'combined_{folder}.csv'
                combined_df.to_csv(output_file, index=False)
                print(f"\n已保存合并后的数据到 {output_file}")
                print(f"合并后的数据形状: {combined_df.shape}")
            except Exception as e:
                print(f"合并数据时出错: {str(e)}")
        else:
            print(f"没有成功读取任何 {folder} 文件夹中的文件")
    print('done!')

if __name__ == "__main__":
    main()

In [ ]:
#####         睡眠质量评估系统构建
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
# from numba import jit
from concurrent.futures import ThreadPoolExecutor
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号
# 1. 数据预处理


def prepare_data(hrv_features, motion_features, labels):
    # 使用多线程并行合并数据
    with ThreadPoolExecutor(max_workers=3) as executor:
        features_future = executor.submit(
            pd.merge_asof,
            hrv_features.sort_values('time'),
            motion_features.sort_values('time'),
            on='time',
            by='user_id',
            direction='nearest',
            tolerance=30
        )
        data_future = executor.submit(
            pd.merge_asof,
            features_future.result().sort_values('time'),
            labels.sort_values('time'),
            on='time',
            by='user_id',
            direction='nearest',
            tolerance=30
        )
        data = data_future.result()

    # 删除缺失值
    data = data.dropna()

    # 特征选择和标准化
    feature_cols = ['sdnn', 'rmssd', 'acc_magnitude',
                    'activity_count', 'activity_intensity']
    target_cols = ['sleep_state']

    scaler = StandardScaler()
    data.loc[:, feature_cols] = scaler.fit_transform(data[feature_cols])

    return data, feature_cols, target_cols, scaler

# # 使用numba加速计算
# @jit(nopython=True)


def calculate_magnitude(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2)


def calculate_activity_features_optimized(motion_data):
    # 创建DataFrame的副本以避免SettingWithCopyWarning
    motion_data = motion_data.copy()
    # 向量化计算加速度
    motion_data.loc[:, 'acc_magnitude'] = calculate_magnitude(
        motion_data['x'].values,
        motion_data['y'].values,
        motion_data['z'].values
    )

    # 使用更高效的滚动窗口计算
    window_size = 30
    motion_data.loc[:, 'activity_count'] = motion_data.groupby('user_id')['acc_magnitude'].transform(
        lambda x: x.rolling(window_size).apply(
            lambda y: np.count_nonzero(y > 0.1), raw=True)
    )
    motion_data.loc[:, 'activity_intensity'] = motion_data.groupby('user_id')['acc_magnitude'].transform(
        lambda x: x.rolling(window_size).mean()
    )

    return motion_data


def calculate_hrv_features_optimized(heart_rate_data):
    # 创建DataFrame的副本以避免SettingWithCopyWarning
    heart_rate_data = heart_rate_data.copy()
    # 向量化计算
    heart_rate_data.loc[:, 'rr_interval'] = 60000 / \
        heart_rate_data['heart_rate'].values

    window_size = 30
    # 预计算差分
    rr_diff = heart_rate_data.groupby('user_id')['rr_interval'].diff().values

    # 并行计算特征
    with ThreadPoolExecutor(max_workers=2) as executor:
        sdnn_future = executor.submit(
            lambda: heart_rate_data.groupby('user_id')['rr_interval'].transform(
                lambda x: x.rolling(window_size).std()
            )
        )
        rmssd_future = executor.submit(
            lambda: np.sqrt(pd.Series(rr_diff).groupby(heart_rate_data['user_id']).rolling(window_size).apply(
                lambda y: np.mean(y**2), raw=True
            ).reset_index(level=0, drop=True))
        )

        heart_rate_data.loc[:, 'sdnn'] = sdnn_future.result()
        heart_rate_data.loc[:, 'rmssd'] = rmssd_future.result()

    return heart_rate_data[['time', 'user_id', 'sdnn', 'rmssd']]


def calculate_sleep_cycle_features_optimized(labels_data):
    # 使用更高效的聚合方式
    sleep_features = labels_data.groupby(['user_id', 'sleep_state'], observed=True).agg(
        min_time=('time', 'min'),
        max_time=('time', 'max')
    ).reset_index()

    sleep_features['duration'] = sleep_features['max_time'] - \
        sleep_features['min_time']

    # 使用更快的过滤方式
    sleep_mask = sleep_features['sleep_state'].between(1, 5)
    sleep_efficiency = sleep_features.groupby('user_id').agg(
        sleep_efficiency=('duration', lambda x: x[sleep_mask].sum() / x.sum()),
        total_sleep_time=('duration', lambda x: x[sleep_mask].sum()),
        total_bed_time=('duration', 'sum')
    ).reset_index()

    # 保留原始的时间数据和睡眠状态
    original_data = labels_data[['user_id',
                                 'time', 'sleep_state']].drop_duplicates()
    sleep_efficiency = pd.merge(sleep_efficiency, original_data, on='user_id')

    return sleep_efficiency

# 2. 优化数据集类


class SleepDataset(Dataset):
    def __init__(self, data, feature_cols, target_cols, sequence_length=30):
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.sequence_length = sequence_length

        # 预计算所有序列
        self.sequences = []
        self.labels = []

        # 使用groupby.apply加速
        def process_group(group):
            # 明确选择需要的列，不包括分组列
            features = group[feature_cols].values
            targets = group[target_cols].values
            seqs = [features[i:i+sequence_length]
                    for i in range(len(features) - sequence_length + 1)]
            lbls = [targets[i+sequence_length-1]
                    for i in range(len(features) - sequence_length + 1)]
            return seqs, lbls

        # 使用group_keys=False来避免分组键
        results = data.groupby(
            'user_id', group_keys=False).apply(process_group)
        for seqs, lbls in results:
            self.sequences.extend(seqs)
            self.labels.extend(lbls)

        # 转换为numpy数组减少内存占用
        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.from_numpy(self.sequences[idx]), torch.from_numpy(self.labels[idx])

# 3. 优化LSTM模型


class SleepLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # 使用双向LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True)
        # 调整全连接层输入大小
        self.fc = nn.Linear(hidden_size * 2, num_classes)

        # 初始化隐藏状态
        self.h0 = nn.Parameter(torch.zeros(num_layers * 2, 1, hidden_size))
        self.c0 = nn.Parameter(torch.zeros(num_layers * 2, 1, hidden_size))

    def forward(self, x):
        # 重用隐藏状态
        batch_size = x.size(0)
        h0 = self.h0.expand(-1, batch_size, -1).contiguous().to(x.device)
        c0 = self.c0.expand(-1, batch_size, -1).contiguous().to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

# 4. 优化训练过程


def train_model(model, train_loader, criterion, optimizer, num_epochs):
    # 使用GPU加速
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # 使用混合精度训练
    scaler = torch.amp.GradScaler('cuda')

    # 使用学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    model.train()
    best_loss = float('inf')

    for epoch in range(num_epochs):
        total_loss = 0.0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad(set_to_none=True)

            # 使用混合精度训练
            with torch.amp.autocast('cuda'):
                output = model(data)
                loss = criterion(output, target.long().squeeze())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        scheduler.step(avg_loss)

        if (epoch+1) % 5 == 0:
            print(
                f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')

        # 保存最佳模型
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), 'best_model.pth')

# 5. 优化评估过程


def evaluate_model(model, test_loader):
    model.eval()
    device = next(model.parameters()).device
    correct = total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target.squeeze()).sum().item()

    accuracy = correct / total
    print(f'Test Accuracy: {accuracy:.4f}')
    return accuracy

# 6. 优化睡眠分析


def analyze_sleep(model, data, feature_cols, sequence_length=30):
    model.eval()
    device = next(model.parameters()).device
    predictions = []

    for user_id, group in data.groupby('user_id'):
        features = group[feature_cols].values
        if len(features) < sequence_length:
            continue

        # 使用 numpy.array 一次性转换序列
        sequences = np.array([features[i:i+sequence_length]
                              for i in range(len(features) - sequence_length + 1)])

        # 预测
        with torch.no_grad():
            inputs = torch.from_numpy(sequences).float().to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            predictions.extend(predicted.cpu().numpy())

    return predictions

# 7. 可视化结果


def visualize_results(data, predictions):
    # 计算睡眠效率
    total_sleep_time = len([p for p in predictions if p > 0])
    total_time = len(predictions)
    sleep_efficiency = total_sleep_time / total_time
    print(f'睡眠效率: {sleep_efficiency:.2f}')
    # 计算各阶段时长
    stage_durations = {}
    for stage in range(6):
        stage_durations[stage] = len([p for p in predictions if p == stage])

    # 绘制睡眠阶段分布
    plt.figure(figsize=(12, 6))
    plt.bar(stage_durations.keys(), stage_durations.values())
    plt.title('睡眠阶段分布')
    plt.xlabel('睡眠阶段')
    plt.ylabel('时长(分钟)')
    plt.xticks(range(6), ['清醒', 'N1', 'N2', 'N3', 'N4', 'REM'])
    plt.savefig('sleep_analysis_results.png')  # 保存图片到当前路径
    plt.close()  # 关闭图形，避免内存泄漏

    return sleep_efficiency, stage_durations

# 主程序


def main():
    # 读取数据并处理
    print("读取数据并处理...")
    df = pd.read_csv('cleaned_sleep_data_fixed_time.csv')
    motion_features = calculate_activity_features_optimized(
        df[['time', 'x', 'y', 'z', 'user_id']])
    hrv_features = calculate_hrv_features_optimized(
        df[['time', 'heart_rate', 'user_id']])
    labels = calculate_sleep_cycle_features_optimized(
        df[['time', 'sleep_state', 'user_id']])
    print("数据处理完成")

    # 准备数据
    print("准备数据...")
    data, feature_cols, target_cols, scaler = prepare_data(
        hrv_features, motion_features, labels)
    print("数据准备完成")

    # 划分训练集和测试集
    print("划分训练集和测试集...")
    train_data, test_data = train_test_split(
        data, test_size=0.2, random_state=42)
    print("划分训练集和测试集完成")

    # 创建数据集
    print("创建数据集...")
    train_dataset = SleepDataset(train_data, feature_cols, target_cols)
    test_dataset = SleepDataset(test_data, feature_cols, target_cols)
    print("创建数据集完成")

    # 创建数据加载器
    print("创建数据加载器...")
    train_loader = DataLoader(train_dataset,
                              batch_size=128,
                              shuffle=True,
                              num_workers=4,
                              pin_memory=True)

    test_loader = DataLoader(test_dataset,
                             batch_size=128,
                             shuffle=False,
                             num_workers=4,
                             pin_memory=True)
    print("创建数据加载器完成")

    # 初始化模型
    print("初始化模型...")
    input_size = len(feature_cols)
    hidden_size = 128
    num_layers = 2
    num_classes = 6

    model = SleepLSTM(input_size, hidden_size, num_layers, num_classes)

    # 加载已训练的最佳模型
    print("加载已训练的最佳模型...")
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()  # 设置为评估模式
    print("模型加载完成")

    # 评估模型
    print("评估模型...")
    accuracy = evaluate_model(model, test_loader)

    # 分析特定用户的睡眠
    print("\n分析特定用户(user_id=1066528)的睡眠...")
    user_specific_data = data[data['user_id'] == 1066528]

    # 分析睡眠
    predictions = analyze_sleep(model, user_specific_data, feature_cols)

    # 可视化结果
    print("可视化特定用户的睡眠结果...")
    sleep_efficiency, stage_durations = visualize_results(
        user_specific_data, predictions)

    print(f"\n用户 1066528 的睡眠分析:")
    print(f"睡眠效率: {sleep_efficiency:.2%}")
    print("\n各阶段时长(分钟):")
    stage_names = ['清醒', 'N1', 'N2', 'N3', 'N4', 'REM']
    for stage, duration in stage_durations.items():
        print(f"{stage_names[stage]}阶段: {duration}")

    # 返回分析结果，方便后续使用
    return {
        'user_id': 1066528,
        'sleep_efficiency': sleep_efficiency,
        'stage_durations': stage_durations
    }


# 运行主程序
if __name__ == "__main__":
    main()

In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'WenQuanYi Micro Hei']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
# 精确读取文件（适配图片中的混乱文件名）
def load_sleep_data(folder_path):
    all_data = []
    for file in os.listdir(folder_path):
        if any(kw in file.lower() for kw in ["label", "sleep"]):  # 兼容所有变体
            try:
                # 提取志愿者ID（处理图片中的"759667__aDEled_sleep.txt"等异常）
                vid = ''.join(filter(str.isdigit, file.split('_')[0]))
                if not vid: continue  # 跳过无效文件名
                
                # 读取数据（图片显示为两列数字，空格分隔）
                df = pd.read_csv(os.path.join(folder_path, file),
                                sep=r"\s+",
                                header=None,
                                names=["seconds", "stage"],
                                engine="python")
                
                # 数据清洗（第二张图显示stage为0-3）
                df = df.dropna().query("stage in [0,1,2,3]")
                df["volunteer_id"] = vid
                df["minutes"] = df["seconds"] / 60
                all_data.append(df)
            except Exception as e:
                print(f"文件{file}处理异常: {str(e)}")
    return pd.concat(all_data)


#####      不同志愿者睡眠阶段比例比较图（无标”!”）
# 计算各志愿者睡眠阶段占比
stage_pct = (sleep_df.groupby(["volunteer_id", "stage"])
                   .size()
                   .unstack()
                   .apply(lambda x: x/x.sum(), axis=1))
# 添加临床注释（示例）
abnormal_ids = stage_pct[stage_pct[0] > 0.3].index  # 清醒占比>30%的志愿者
for vid in abnormal_ids:
    plt.text(stage_pct.columns.get_loc(0)+0.3, 
             stage_pct.index.get_loc(vid),
             "!", color='red', fontsize=14)
    
plt.figure(figsize=(10,6))
sns.heatmap(stage_pct, cmap="YlGnBu", annot=True, fmt=".1%", cbar=False)
plt.title("不同志愿者的睡眠阶段比例比较（只显示0-3阶段）", pad=20)
plt.xlabel("睡眠阶段编码")
plt.ylabel("志愿者id")
plt.show()


#####        睡眠阶段随时间变化趋势图（有标”!”）
# 临床异常标记（直接在热力图上标注）,使问题数据更加明显
plt.figure(figsize=(12,6))
ax = sns.heatmap(stage_pct, cmap="YlGnBu", annot=True, fmt=".1%", cbar=False)

# 标记异常值（示例：清醒占比>25%的志愿者）
abnormal_ids = stage_pct[stage_pct[0] > 0.25].index
for vid in abnormal_ids:
    # 获取坐标位置（列索引+0.3是为了让标记显示在单元格右侧）
    col_idx = stage_pct.columns.get_loc(0) + 0.3
    row_idx = stage_pct.index.get_loc(vid)
    ax.text(col_idx, row_idx, "!!!出现问题", 
            color='red', fontsize=14, ha='center')

plt.title("Sleep Stage Proportions with Clinical Annotations", pad=20)
plt.show()

In [ ]:
#####       总睡眠周期（单个）
def plot_sleep_cycles(vid):
    df = sleep_df[sleep_df["volunteer_id"]==vid].sort_values("seconds")
    
    # 检测周期起点
    n2_periods = df[df["stage"]==2]["minutes"].values
    peaks, _ = find_peaks(n2_periods, distance=30)
    
    # 创建画布和样式设置
    plt.figure(figsize=(16, 5), dpi=120)
    ax = plt.gca()
    
    # 1. 背景增强
    ax.set_facecolor('#F5F5F5')
    plt.grid(color='white', linestyle='--', alpha=0.7)
    
    # 2. 睡眠阶段区域着色
    stage_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#A37EBA']
    for stage in range(4):
        ax.axhspan(stage-0.5, stage+0.5, color=stage_colors[stage], alpha=0.1)
    
    # 3. 折线图优化
    plt.plot(df["minutes"], df["stage"], 
             color='#333333', 
             alpha=0.7, 
             linewidth=1.5,
             label="Sleep pattern")
    
    # 4. 周期起点标记增强
    plt.scatter(n2_periods[peaks], [2]*len(peaks), 
                color='#FF2D2D', 
                s=80,
                edgecolor='white',
                linewidth=1,
                zorder=3,
                label=f"Cycle start (detected: {len(peaks)})")
    
    # 5. 坐标轴美化
    ax.set_yticks(range(4))
    ax.set_yticklabels(["醒着", "N1睡眠期", "N2睡眠期", "深度睡眠"], fontsize=12)
    ax.set_xlabel("记录时间（分钟）", fontsize=12, labelpad=10)
    ax.set_ylabel("睡眠阶段", fontsize=12, labelpad=10)
    
    # 6. 标题和图例
    plt.title(f"睡眠架构分析-志愿者 {vid}\n"
             f"检测到的总睡眠周期：{len(peaks)}", 
             pad=20, fontsize=14, fontweight='bold')
    
    legend = plt.legend(loc='upper right', framealpha=1, edgecolor='#DDDDDD')
    legend.get_frame().set_facecolor('#FAFAFA')
    
    # 7. 边框优化
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    plt.tight_layout()
    plt.show()

    # 深睡眠占比
    deep_ratio = len(df[df['stage']==3])/len(df)
    # 觉醒次数
    awakening = len(df[df['stage'].diff()==-0])  # 从非清醒→清醒的跳变
    print(f"深睡眠占比:{deep_ratio},觉醒次数:{awakening}")

plot_sleep_cycles("759667")

In [ ]:
#####       睡眠阶段出现时间分布图
plt.figure(figsize=(10, 6), facecolor='white')

# 箱线图主体（保持原样式）
boxprops = dict(linewidth=2, edgecolor='black')
medianprops = dict(linewidth=2.5, color='black')
ax = sns.boxplot(data=sleep_df, x="stage", y="minutes",
                palette=["#E74C3C", "#4ECDC4", "#3498DB", "#9B59B6"],  # 红/青绿/蓝/紫
                width=0.7,
                boxprops=boxprops,
                medianprops=medianprops,
                flierprops=dict(marker='o', markersize=5))

# 坐标轴设置（全中文）
plt.yticks(np.arange(0, 481, 60), fontsize=10)
plt.xticks([0, 1, 2, 3], 
          ["清醒\n(阶段0)", "浅睡\n(N1)", "浅睡\n(N2)", "深睡\n(N3)"],  # 中文标签
          fontsize=11, linespacing=1.2)
plt.ylabel("记录开始时间（分钟）", fontsize=12, labelpad=10)  # 中文
plt.xlabel("")  # 移除x轴标题

# 中文标题和说明
plt.title("睡眠阶段出现时间分布\n（显示夜间各阶段发生时间）",  # 中文标题
         fontsize=13, pad=20, loc='left')

# 中文说明文字
plt.text(0.5, -0.12, 
        "箱体显示25-75百分位范围\n水平线表示中位数值",  # 中文说明
        transform=ax.transAxes, ha='center', fontsize=10,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

# 网格和边框（保持原样）
ax.grid(axis='y', linestyle='--', alpha=0.4, color='gray')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()